# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## A logistic regression model, built from parts (in class)

In the LELA60331 seminars you have built **logistic regression** step by step, as a long series of notebook cells, and in Week 7 you trained it on simulated data. In this workbook you will build the same classifier again, using what you learned about classes in Week 7, in a way that makes it easy to reuse and to extend. It has three parts:

| Part | What it does |
|---|---|
| a **layer** (the class `Linear`) | turns each example's features into a score (the **forward** pass), works out the gradients for its weights (the **backward** pass), and updates its weights |
| a **loss** (two functions, `bce_loss` and `bce_grad`) | measures how wrong the scores are, and says which way each score should move |
| a **training loop** | puts them together |

By the end of this workbook you will train the model with a loop like this one, in which each line does one job:

```python
for epoch in range(n_epochs):
    scores = model.forward(X)                 # the forward pass
    loss = bce_loss(scores, y)                # how wrong is it?
    model.backward(bce_grad(scores, y))       # the backward pass: the gradients
    model.update(lr)                          # change the weights
```

We build every part in small steps: first by working it out for a tiny example, then by adding it to the class.

This workbook is for the class, and takes about an hour. A second workbook, `LELA60341_Wk8_after_class`, has exercises to do on your own afterwards: using and evaluating the model you have trained, and comparing several models.

| Part | Topic |
|---|---|
| 1 | From a script to a class |
| 2 | The data as a matrix |
| 3 | A linear layer, one method at a time |
| 4 | The loss |
| 5 | Training: one step, then many |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · From a script to a class

In the LELA60331 seminars, each step of the classifier is a separate cell, and the cells share their results through **global names**. Even for the simulated data in LELA60331 Week 7 there are a lot of them:

| Name | What it holds |
|---|---|
| `x`, `y`, `X` | the features and the labels |
| `weights`, `bias` | the model's parameters |
| `lr`, `n_epochs` | the training settings |
| `q`, `errors`, `loss_history` | the probabilities, the errors and the losses |
| `score`, `sigmoid`, `average_loss`, `mat_vec`, ... | the functions that work on them |

This works, but it is fragile:

* Run the cells in the wrong order, or re-run one by mistake, and the names hold the wrong values. (In LELA60331 Week 7, `q` holds the probabilities from whichever cell you ran last: the untrained model or the trained one.)
* You can't easily have **two** models at once, because they would both want the name `weights`.
* To use the classifier on new data, you have to remember which cells to copy, and in which order.

A class solves these problems. Its **instance variables** keep the model's data together (its weights, its bias and their gradients), and its **methods** are the operations on them (Week 7). Each model is one object, with its own weights.

### 🏋️ Exercise 1 · Practice — Data or method?

The layer needs each of the things below. For each one, say whether it should be stored as an `"instance variable"` or written as a `"method"`.

In [ ]:
design = {
    "the weights": ???,
    "calculate the score of each example": ???,
    "the gradients for the weights": ???,
    "change the weights by gradient descent": ???,
}

In [ ]:
# ✅ Check
check("data", lambda: design["the weights"] == design["the gradients for the weights"] == "instance variable")
check("operations", lambda: design["calculate the score of each example"] == design["change the weights by gradient descent"] == "method")

---
# Part 2 · The data as a matrix

Here is the simulated data from LELA60331 Weeks 5 and 7: 50 texts, each with two features (the log counts of positive and of negative words) and a label (1 = positive, 0 = negative). As in the seminar, `x[0]` holds feature 1 for every text and `x[1]` holds feature 2. Run the cell.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
print(len(y), "texts,", sum(y), "of them positive")

In LELA60331 Week 7, Part 4 you saw that it is more convenient to store the data as a **matrix** `X` with **one row per text** and one column per feature, so that one matrix-vector multiplication gives the score of every text. You also saw that the gradients need the **transpose** of `X`: the same numbers with the rows turned into columns, one row per feature.

The next cell makes `X` and the `transpose` function, as you did in the seminar (LELA60331 Week 7, Activities 6, 7 and 12), and the helper functions `dot` (Week 4) and `mat_vec` (LELA60331 Week 7, Activity 10). It also defines the `sigmoid` function from LELA60331 Week 5. Run it.

In [ ]:
X = [[x[0][k], x[1][k]] for k in range(len(y))]          # one row [x1, x2] per text

def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

def dot(a, b):
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

print(len(X), "rows;", X[0])
print(transpose([[1, 2, 3], [4, 5, 6]]))

---
# Part 3 · A linear layer, one method at a time

A `Linear` layer does the matrix-vector multiplication from LELA60331 Week 7. It has `n_out` **outputs**, and for each example it gives one score per output. Logistic regression needs just one output: the score that the text is positive. (Other models need more, and that is why the layer allows several.)

We build it in four steps. At each step you work out a small example first, then add one method to the class:

| Step | Method | What it does |
|---|---|---|
| 1 | `__init__` | makes the weights and biases *(done for you)* |
| 2 | `forward(X)` | the scores of a list of examples |
| 3 | `backward(grad)` | the gradients for the weights and biases |
| 4 | `update(lr)` | changes the weights and biases by gradient descent |

## Step 1 · Making a layer

Here is the class with just its constructor. A layer with `n_in` inputs and `n_out` outputs has:

* `self.weight`: a matrix with **one row per output** and one column per input feature
* `self.bias`: a list with one bias per output

Both start at random numbers between 0 and 1, as in LELA60331 Week 7. Run the cell.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

layer = Linear(2, 3)
print(layer.weight)
print(layer.bias)

### 🏋️ Exercise 2 · Practice — Looking at a layer

🔮 **Predict** first: `Linear(4, 2)` has 4 inputs and 2 outputs. How many rows will its weight matrix have, and how many numbers in each row? How many biases?

1. `big`: a `Linear` layer with 4 inputs and 2 outputs
2. `n_rows`, `n_columns` and `n_biases`: the number of rows of `big.weight`, the number of numbers in its first row, and the number of biases

In [ ]:
big = ???                         # 1.
n_rows = ???                   # 2.
n_columns = ???
n_biases = ???
print(n_rows, n_columns, n_biases)           # you should get 2 4 2

In [ ]:
# ✅ Check
check("1. big", lambda: isinstance(big, Linear))
check("2. the shapes: one row of weights per output, one weight per input", lambda: (n_rows, n_columns, n_biases) == (2, 4, 2))

## Step 2 · The forward pass

For one example `x`, the score of output `c` is the dot product of row `c` of the weights with `x`, plus bias `c`. `mat_vec(self.weight, x)` gives all the dot products at once, one per output; then we add each output's bias.

Here is a worked example, with weights and biases chosen by hand so that the numbers are easy:

| output `c` | weights `weight[c]` | bias `bias[c]` | score for `x = [2, 3]` |
|---|---|---|---|
| 0 | `[1, 0]` | 0 | 1 × 2 + 0 × 3 + 0 = **2** |
| 1 | `[0, 1]` | 10 | 0 × 2 + 1 × 3 + 10 = **13** |
| 2 | `[1, 1]` | 100 | 1 × 2 + 1 × 3 + 100 = **105** |

### 🏋️ Exercise 3 · Practice — The scores of one example, by hand

Using the weights and biases from the table:

1. `sums`: `mat_vec` with `W` and `x` (one number per output)
2. `z`: complete the loop so that it adds each output's bias

In [ ]:
W = [[1, 0], [0, 1], [1, 1]]
b = [0, 10, 100]
x_example = [2, 3]

sums = ???                 # 1.
z = []
for c in range(len(sums)):
    z.append(???)                 # 2.
print(sums, z)                                 # you should get [2, 3, 5] [2, 13, 105]

In [ ]:
# ✅ Check
check("1. sums", lambda: sums == [2, 3, 5])
check("2. z", lambda: z == [2, 13, 105])

### 🏋️ Exercise 4 · Practice — Adding `forward` to the class

Now put the same steps in a method. `forward(X)` takes a **list of examples** (one row each, like `X`) and returns a matrix with **one row per example** and one score per output. It also keeps `X` in `self.inputs`, because the backward pass will need it.

The cell repeats the class from Step 1, with the new method added. Fill in the blanks 1–2, the same calculations as in Exercise 3 but with `self.weight` and `self.bias`, then run the cell to replace the old class with the new one.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X                                    # kept for the backward pass
        scores = []
        for x in X:                                        # each example
            z = ???                  # 1. one number per output
            for c in range(len(z)):
                z[c] += ???                     # 2. plus that output's bias
            scores.append(z)
        return scores


layer = Linear(2, 3)
layer.weight = [[1, 0], [0, 1], [1, 1]]         # set the weights and biases by hand, to test it
layer.bias = [0, 10, 100]
print(layer.forward([[2, 3], [1, 1]]))           # you should get [[2, 13, 105], [1, 11, 102]]

In [ ]:
# ✅ Check
def _layer():
    l = Linear(2, 3)
    l.weight = [[1, 0], [0, 1], [1, 1]]
    l.bias = [0, 10, 100]
    return l
check("1–2. forward, one example", lambda: _layer().forward([[2, 3]]) == [[2, 13, 105]])
check("1–2. forward, two examples", lambda: _layer().forward([[2, 3], [1, 1]]) == [[2, 13, 105], [1, 11, 102]])
check("forward keeps the inputs", lambda: (lambda l: (l.forward([[2, 3]]), l.inputs)[1])(_layer()) == [[2, 3]])

## Step 3 · The backward pass

The backward pass works out the gradients for the weights and biases. It is given `grad`, which has the same shape as the scores: `grad[k][c]` says how much the loss changes when the score of output `c` for example `k` goes up. (In Part 4 you will see where `grad` comes from: it is the error from LELA60331 Week 5.)

The rule is the one from LELA60331 Week 7:

* the gradient for `weight[c][j]` is `grad[k][c] * inputs[k][j]`: the error for that output, times the feature
* the gradient for `bias[c]` is `grad[k][c]`

With several examples, each example's contribution is **added up**.

Here is a worked example for **one** example, `x = [2, 3]`, whose gradients for the three outputs are `[1, 0, 2]`:

| | feature 0 (`x[0]` = 2) | feature 1 (`x[1]` = 3) | bias |
|---|---|---|---|
| output 0 (grad 1) | 1 × 2 = **2** | 1 × 3 = **3** | **1** |
| output 1 (grad 0) | 0 × 2 = **0** | 0 × 3 = **0** | **0** |
| output 2 (grad 2) | 2 × 2 = **4** | 2 × 3 = **6** | **2** |

### 🏋️ Exercise 5 · Practice — The gradients for one example, by hand

Make the gradient matrix in the table with nested loops: an outer loop for the rows (one per output) and an inner loop for the columns (one per feature), as for matrices in Week 4:

1. complete the inner loop, so that row `c` holds `g[c] * x_example[j]` for each feature `j`
2. `bias_grad_example`: the gradients for the biases. *Hint:* they are just the gradients `g`; `list(g)` makes a copy.

In [ ]:
x_example = [2, 3]
g = [1, 0, 2]                     # the gradient of each of the three outputs, for this example

weight_grad_example = []
for c in range(len(g)):           # each output: one row
    row = []
    for j in range(len(x_example)):              # each feature: one column
        row.append(???)        # 1.
    weight_grad_example.append(row)
bias_grad_example = ???                    # 2.
print(weight_grad_example, bias_grad_example)    # you should get [[2, 3], [0, 0], [4, 6]] [1, 0, 2]

In [ ]:
# ✅ Check
check("1. weight_grad_example", lambda: weight_grad_example == [[2, 3], [0, 0], [4, 6]])
check("2. bias_grad_example", lambda: bias_grad_example == [1, 0, 2])

### 🏋️ Exercise 6 · Practice — Adding `backward` to the class

In the method, the gradients start as 0s, and each example's contribution is **added** to them, so that with several examples we get the total. There is one loop for the examples, one for the outputs and one for the features. Fill in the blanks 1–2: the same calculations as in Exercise 5, with `grad[k][c]` and `self.inputs[k][j]` for example `k`.

(The method's last line returns the gradients for the layer's **inputs**. They are needed when one layer feeds into another. It is done for you.)

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X
        scores = []
        for x in X:
            z = mat_vec(self.weight, x)
            for c in range(len(z)):
                z[c] += self.bias[c]
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]     # start from 0s
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):                  # each example
            for c in range(n_out):                         # each output
                for j in range(n_in):                      # each input feature
                    self.weight_grad[c][j] += ???     # 1.
                self.bias_grad[c] += ???                                  # 2.
        # the gradients for the inputs, for a layer that comes before this one (done for you)
        return [mat_vec(transpose(self.weight), gr) for gr in grad]


layer = Linear(2, 3)
layer.weight = [[1, 0], [0, 1], [1, 1]]
layer.bias = [0, 10, 100]
layer.forward([[2, 3]])
layer.backward([[1, 0, 2]])
print(layer.weight_grad, layer.bias_grad)        # the same as Exercise 5: [[2, 3], [0, 0], [4, 6]] [1, 0, 2]
layer.forward([[2, 3], [1, 1]])
layer.backward([[1, 0, 0], [0, 0, 2]])
print(layer.weight_grad, layer.bias_grad)        # you should get [[2, 3], [0, 0], [2, 2]] [1, 0, 2]

In [ ]:
# ✅ Check
def _layer():
    l = Linear(2, 3)
    l.weight = [[1, 0], [0, 1], [1, 1]]
    l.bias = [0, 10, 100]
    return l
def _one():
    l = _layer(); l.forward([[2, 3]]); l.backward([[1, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [4, 6]] and l.bias_grad == [1, 0, 2]
check("1–2. backward, one example: the same as Exercise 5", _one)
def _two():
    l = _layer(); l.forward([[2, 3], [1, 1]]); l.backward([[1, 0, 0], [0, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [2, 2]] and l.bias_grad == [1, 0, 2]
check("1–2. backward, two examples: their contributions are added", _two)
def _twice():
    l = _layer(); l.forward([[2, 3]]); l.backward([[1, 0, 2]]); l.backward([[1, 0, 2]])
    return l.weight_grad == [[2, 3], [0, 0], [4, 6]]
check("backward starts from 0s each time", _twice)

Look at the second example: for output 2, example 0's gradient is 0 and example 1's is 2, so the gradients for output 2's weights are 0 × `[2, 3]` + 2 × `[1, 1]` = `[2, 2]`.

## Step 4 · The update

Finally, gradient descent, exactly as in LELA60331 Weeks 4, 5 and 7: **new value = old value − learning rate × gradient**, for every weight and every bias.

### 🏋️ Exercise 7 · Practice — Adding `update` to the class

Fill in the blanks 1–2. With a learning rate of 0.5 and the gradients from the second example in Exercise 6, weight `[0][1]` becomes 0 − 0.5 × 3 = −1.5, and bias 2 becomes 100 − 0.5 × 2 = 99.0.

In [ ]:
class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X
        scores = []
        for x in X:
            z = mat_vec(self.weight, x)
            for c in range(len(z)):
                z[c] += self.bias[c]
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):
            for c in range(n_out):
                for j in range(n_in):
                    self.weight_grad[c][j] += grad[k][c] * self.inputs[k][j]
                self.bias_grad[c] += grad[k][c]
        return [mat_vec(transpose(self.weight), gr) for gr in grad]

    def update(self, lr):
        for c in range(len(self.weight)):
            for j in range(len(self.weight[c])):
                self.weight[c][j] -= ???          # 1.
            self.bias[c] -= ???                        # 2.


layer = Linear(2, 3)
layer.weight = [[1, 0], [0, 1], [1, 1]]
layer.bias = [0, 10, 100]
layer.forward([[2, 3], [1, 1]])
layer.backward([[1, 0, 0], [0, 0, 2]])
layer.update(0.5)
print(layer.weight, layer.bias)        # you should get [[0.0, -1.5], [0.0, 1.0], [0.0, 0.0]] [-0.5, 10.0, 99.0]

In [ ]:
# ✅ Check
def _update():
    l = Linear(2, 3)
    l.weight = [[1, 0], [0, 1], [1, 1]]
    l.bias = [0, 10, 100]
    l.forward([[2, 3], [1, 1]])
    l.backward([[1, 0, 0], [0, 0, 2]])
    l.update(0.5)
    return l.weight == [[0.0, -1.5], [0.0, 1.0], [0.0, 0.0]] and l.bias == [-0.5, 10.0, 99.0]
check("1–2. update", _update)

That's the whole layer: four methods, each of which you have tested on its own.

---
# Part 4 · The loss

The loss measures how wrong the model's scores are. For two classes we use the cross-entropy loss from LELA60331 Week 5 (sometimes called **binary cross-entropy**, or BCE). A layer with one output gives one row per example with one score in it, so the score of example `k` is `scores[k][0]`.

For each example, the loss turns the score into a probability with `sigmoid`, then takes minus the log of the probability given to the correct label. The loss for all the examples is the average.

### 🏋️ Exercise 8 · Practice — The loss for one example, then for all

1. `q_example`: the probability for a score of 2.0
2. `loss_if_negative`: the loss if that example's label is 0. *Hint:* the probability given to the label 0 is `1 - q_example`.
3. Complete `bce_loss(scores, y)`, which returns the average loss over all the examples. The `if` and `else` are done for you.

In [ ]:
q_example = ???                                     # 1.
loss_if_negative = ???                 # 2.
print(round(q_example, 3), round(loss_if_negative, 3))         # you should get 0.881 3.069

def bce_loss(scores, y):
    eps = 0.00001          # a tiny number, so that we never take the log of 0
    losses = []
    for k in range(len(y)):
        q = ???                           # 3. the probability
        if y[k] == 1:
            losses.append(-math.log2(q + eps))
        else:
            losses.append(-math.log2(1 - q + eps))
    return ???                        #    the average

print(round(bce_loss([[0.0], [2.0]], [1, 0]), 3))              # you should get 2.034: the average of 1.0 and 3.069

In [ ]:
# ✅ Check
check("1. q_example", lambda: round(q_example, 3) == 0.881)
check("2. loss_if_negative", lambda: round(loss_if_negative, 3) == 3.069)
check("3. bce_loss", lambda: round(bce_loss([[0.0], [2.0]], [1, 0]), 3) == 2.034)

### The gradient of the loss

The backward pass needs one more thing from the loss: for each example, how much the loss changes when that example's score goes up. For this loss, that is exactly the **error** from LELA60331 Week 5, divided by the number of examples (because the loss is an average):

`(q - y[k]) / n`, where `q` is the sigmoid of example `k`'s score

For the two examples above (scores 0.0 and 2.0, labels 1 and 0):

| example | `q` | label | `(q - label) / 2` |
|---|---|---|---|
| 0 | 0.5 | 1 | −0.25 |
| 1 | 0.881 | 0 | 0.44 |

`bce_grad` returns these as a matrix with one row per example and one number in each row, the same shape as the scores, so that it can be passed straight to the layer's `backward`.

### 🏋️ Exercise 9 · Practice — `bce_grad`

Complete the loop in `bce_grad(scores, y)`: for each example, add a row with one number, `(q - y[k]) / n`.

In [ ]:
def bce_grad(scores, y):
    n = len(y)
    grad = []
    for k in range(n):
        q = sigmoid(scores[k][0])
        grad.append(???)          # a row with one number
    return grad

print([[round(v, 3) for v in row] for row in bce_grad([[0.0], [2.0]], [1, 0])])   # you should get [[-0.25], [0.44]]

In [ ]:
# ✅ Check
check("bce_grad", lambda: [[round(v, 3) for v in row] for row in bce_grad([[0.0], [2.0]], [1, 0])] == [[-0.25], [0.44]])
check("one row per example, with one number", lambda: [len(row) for row in bce_grad([[0.0], [2.0], [1.0]], [1, 0, 1])] == [1, 1, 1])

---
# Part 5 · Training: one step, then many

Logistic regression is just a `Linear` layer with one output, trained with this loss. As in LELA60331 Week 5, we first make **one** improvement, step by step, and check that the loss goes down. Then we repeat it many times.

### 🏋️ Exercise 10 · Practice — One step

1. `model`: a `Linear` layer with 2 inputs and 1 output (after `random.seed(10)`, so that everyone gets the same starting weights as in LELA60331 Week 7)
2. `scores`: the scores of every text (the forward pass)
3. `loss_before`: the loss
4. the backward pass: give `model.backward` the gradients from `bce_grad`
5. the update, with a learning rate of 0.5
6. `loss_after`: the loss of the updated model. *Hint:* you need new scores.

In [ ]:
random.seed(10)
model = ???                                  # 1.
scores = ???                             # 2.
loss_before = ???                     # 3.
???                   # 4.
???                                     # 5.
loss_after = ???            # 6.
print(round(loss_before, 3), round(loss_after, 3))      # you should get 2.455 0.866

In [ ]:
# ✅ Check
check("1. one output, two inputs", lambda: len(model.weight) == 1 and len(model.weight[0]) == 2)
check("3. loss_before", lambda: abs(loss_before - 2.455) < 0.001)
check("6. the loss went down", lambda: loss_after < loss_before)

### 🏋️ Exercise 11 · Practice — The training loop

Now repeat the step many times. Each pass through the data (each **epoch**) does the same four things as Exercise 10:

1. calculates the scores (the **forward pass**)
2. calculates the loss, and records it
3. calculates the gradients (the **backward pass**)
4. updates the weights and the bias

The settings are those of LELA60331 Week 7: a learning rate of 0.5 and 1000 epochs, with a new model made after `random.seed(10)`.

In [ ]:
random.seed(10)
model = Linear(2, 1)
lr = 0.5
n_epochs = 1000
loss_history = []

for epoch in range(n_epochs):
    scores = ???                  # 1.
    loss = ???                 # 2.
    loss_history.append(loss)
    ???        # 3.
    ???                           # 4.

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print(model.weight, model.bias)                # about [[1.17, -1.32]] and [0.51]
print("final loss:", round(loss_history[-1], 3))    # you should get 0.656

In [ ]:
# ✅ Check
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("the first loss is the one from Exercise 10", lambda: abs(loss_history[0] - loss_before) < 1e-12)
check("the same weights as LELA60331 Week 7", lambda: [round(w, 2) for w in model.weight[0]] == [1.17, -1.32] and round(model.bias[0], 2) == 0.51)
check("final loss about 0.656", lambda: abs(loss_history[-1] - 0.656) < 0.001)

These are the weights from LELA60331 Week 7. Compare the two pieces of code: which lines in the seminar's training loop does each line of this loop replace?

---
## ✏️ Summary

You can now:

* explain why a program with many shared global names is fragile, and how a class helps
* build a class one method at a time, testing each method on a small example
* write a `Linear` layer with a **forward** pass, a **backward** pass and an **update**
* write a loss as two functions: one for its value, and one for its gradient
* make one training step by hand, then train a model with a loop of four lines: forward, loss, backward, update

**After class:** work through `LELA60341_Wk8_after_class`, which uses the model you have built here.